# Data Preprocessing and InstanSeg Evaluation

This notebook is dedicated to preprocessing microscopy images (brightfield and fluorescent) and their corresponding masks for the InstanSeg model.
Raw data format inconsistencies are addressed here before model evaluation.

**Note:** you can run this code with any image formats. Most examples, use `tiff` images, as they are less likely to get any errors from InstanSeg during training or testing. However, you can experiment with other data types, keeping in mind that it may sometimes cause unexpected errors.

**How to use this notebook?**

1. Make a copy of this notebook.
2. Load some images into the files space.
3. Always run `Section 1` to setup the environment.
4. Run the code cells you need, changing the paths so that they refer to your images and/or folders.
5. After making some changes to your image, you may want to pass it to one of the InstanSeg models in `Section 4`, and evaluate the effectiveness of a specific preprocessing technique.
6. Use `Section 5` for visualizing any image or mask if needed.

## 1. Environment Setup & Model Initialization
First, we install required dependencies and load core libraries. We then initialize the pre-trained `InstanSeg` models for both brightfield and fluorescence microscopy images.

In [ ]:
!pip install monai

In [ ]:
!pip install instanseg-torch

In [ ]:
!pip install imagecodecs

In [ ]:
# Deep Learning and Segmentation
import torch
import torchvision
from instanseg import InstanSeg
from instanseg.utils.utils import show_images

# Image Processing and Math
import numpy as np
import tifffile as tiff
from PIL import Image, ImageEnhance
from skimage import io
import cv2
import matplotlib.pyplot as plt

# Initialize Models
instanseg_brightfield = InstanSeg("brightfield_nuclei", verbosity=1, image_reader="skimage.io")
instanseg_fluorescence = InstanSeg("fluorescence_nuclei_and_cells", verbosity=1, image_reader="skimage.io")

## 2. Format Standardization
Microscopy images and CVAT annotations often come in mixed formats. In this section, we standardize our data:
1. Converting standard images (PNG/JPG) to TIFF formats.
2. Fixing bit-depth issues (e.g., converting unsupported `int8` TIFFs to standard `uint8`).
3. Ensuring images have the expected 3 channels (RGB) required by some processes.

In [ ]:
# change the order of channels

im = Image.open('your-image-path')
img = im.convert("RGB")
im.save("your-save-path.tif", 'TIFF')

In [ ]:
# convert to "normal" tiff: int8 -> uint8

input_path = 'your-input-path.tif'
output_path = 'your-output-path.tif'

image_data_int8 = tiff.imread(input_path)
image_data_uint8 = image_data_int8.astype(np.uint8)
tiff.imwrite(output_path, image_data_uint8)

with Image.open(output_path) as im:
    print("Success! The new file can now be opened by Pillow.")
    print(f"   -> Image Mode: {im.mode}, Size: {im.size}")

In [ ]:
# the input image must have 3 channels

# convert to grayscale
image = Image.open("/content/img_212.TIF").convert("L")
image_array = np.array(image)

# duplicate and stack the 2D matrix to get H x W x 3
rgb_array = np.stack([image_array] * 3, axis = -1)

rgb_image = Image.fromarray(rgb_array, mode="RGB")
rgb_image.save("/content/example.tif")

## 3. Image Resizing and Enhancement
InstanSeg processes images efficiently at specific resolutions. Here, we check the native dimensions of our cryobiology samples and downscale them (* e.g., towards 512x512) to fit memory and model requirements. We also test contrast enhancement to make cellular structures more visible.

\* All parameters are usually tuned manually, and the optimal ones are identified empirically.

In [ ]:
# print size

image = Image.open("your-image-path.tif")
width, height = image.size
print(f"{width}x{height}")

In [ ]:
# changing size

image = Image.open("your-image-path.tif")

# experiment with different parameters
w, h = image.width, image.height
scale = max(1, w // 512, h // 512)
new_size = (image.width // scale, image.height // scale)

# you may try both options below - and choose the best
# image_resized = image.resize(new_size, Image.LANCZOS)
image_resized = image.resize(new_size, Image.NEAREST)

image_resized.save("your-save-path.tif")

In [ ]:
# contrast

image = Image.open("your-image-path.jpg")
enhancer = ImageEnhance.Contrast(image)
image_contrasted = enhancer.enhance(2)
image_contrasted.save("your-save-path.jpg")

In [ ]:
# changing color - not needed in most cases
# add red color

image = Image.open("your-image-path.jpg").convert("RGB")
red_layer = Image.new("RGB", image.size, (255, 0, 0))
red_image = Image.blend(image, red_layer, alpha=0.5)
red_image.save("your-save-path.jpg")

## 4. InstanSeg Model Evaluation
With the data preprocessed, we pass our samples through the InstanSeg models. We evaluate both the fluorescence and brightfield modalities, extracting the number of detected cells and visualizing the predicted segmentation masks.

In [ ]:
# Fluorescence pretrained model
# unexpectedly shows better results (in brightfield images)

image_array, pixel_size = instanseg_fluorescence.read_image("your-image-path.tif")
print(pixel_size)
print(image_array.shape)

labeled_output, image_tensor  = instanseg_fluorescence.eval_small_image(image_array, pixel_size,
                                                                               return_image_tensor = True)

# Cell calculation - custom function - implemented in fork repository
# !pip install git+https://github.com/sonyalytv/instanseg_cryobiology.git
# labeled_output, image_tensor, cells  = instanseg_fluorescence.eval_small_image(image_array, pixel_size,
#                                                                                return_image_tensor = True)
# print("Number of cells detected: ", cells)

print(labeled_output.shape)
print(labeled_output.dtype)
display = instanseg_fluorescence.display(image_tensor, labeled_output)

show_images(display, colorbar=False)

In [ ]:
# Brightfield pretrained model
# Works better for nuclei segmentation

image_array, pixel_size = instanseg_brightfield.read_image("your-image-path.tif")

print(pixel_size)
print(image_array.shape)

labeled_output, image_tensor = instanseg_brightfield.eval_small_image(image_array,
                                                                      pixel_size,
                                                                      resolve_cell_and_nucleus=True,
                                                                      cleanup_fragments = True,
                                                                      target = "cells",
                                                                     verbosity = 0)

display = instanseg_brightfield.display(image_array, labeled_output)

show_images(display, colorbar=False)

## 5. Verification and Diagnostics
Finally, we provide utility blocks to inspect specific output files and verify the integrity of the segmentation masks.

In [ ]:
# take a look at the mask or image - for visual inspection
img = io.imread("your-mask-path.png")
plt.imshow(img)
img.shape

In [ ]:
# check mask values
img = cv2.imread("your-mask-path.tif", cv2.IMREAD_UNCHANGED)

print(f"Unique values found: {np.unique(img)}")